# Three classifier architectures with the fixed JSON retriever

This notebook uses the same precomputed JSON retriever output for all three classifier architectures, so the comparison focuses on the classifier architecture rather than retrieval differences.

## 1. Setup

In [29]:
# If running on Colab, mount Google Drive first.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import json
import re
import os
import random
from pathlib import Path
from typing import Dict, List, Tuple, Callable
from collections import Counter

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

LABEL2ID = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


## 2. Paths

In [31]:
DATA_DIR = Path("/content/drive/MyDrive/NLP/data")
OUTPUT_DIR = Path("/content/drive/MyDrive/NLP/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_PATH = DATA_DIR / "train-claims.json"
DEV_PATH = DATA_DIR / "dev-claims.json"
TEST_PATH = DATA_DIR / "test-claims-unlabelled.json"

# Fixed retriever output for train/dev claims.
TRAIN_RETRIEVER_JSON_PATH = DATA_DIR / "train-retriever-only-k4-bm25200.json"
DEV_RETRIEVER_JSON_PATH = DATA_DIR / "dev-retriever-only-k4-bm25200.json"

# Optional test retriever
TEST_RETRIEVER_JSON_PATH = DATA_DIR / "test-retriever-only-k4-bm25200.json"

## 3. Reproducibility and utility functions

In [32]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def normalise_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%°]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenise(text: str) -> List[str]:
    return normalise_text(text).split()


def concatenate_evidence(evidence_ids: List[str], evidence_corpus: Dict[str, str], max_evidence: int = 4) -> str:
    evidence_texts = []
    for eid in evidence_ids[:max_evidence]:
        if eid in evidence_corpus:
            evidence_texts.append(evidence_corpus[eid])

    if len(evidence_texts) == 0:
        return "No relevant evidence found."

    return " ".join(evidence_texts)

## 4. Load data

In [33]:
evidence_corpus = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_PATH)
dev_claims = load_json(DEV_PATH)

print("Evidence passages:", len(evidence_corpus))
print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))

Evidence passages: 1208827
Train claims: 1228
Dev claims: 154


## 5. Fixed JSON retriever

In [34]:
class FixedJSONRetriever:
    """Use a precomputed retriever-output JSON file as the retriever.

    Expected format:
    {
      "claim-752": {
        "claim_text": "...",
        "claim_label": "SUPPORTS",
        "evidences": ["evidence-1", "evidence-2", ...]
      }
    }
    """

    def __init__(self, retriever_json_path: Path):
        self.retriever_json_path = Path(retriever_json_path)
        self.retriever_results = load_json(self.retriever_json_path)

        # Fallback lookup for cases where only claim_text is passed.
        self.text_to_evidences = {}
        for claim_id, item in self.retriever_results.items():
            claim_text = item.get("claim_text", "")
            self.text_to_evidences[normalise_text(claim_text)] = item.get("evidences", [])

        print("Loaded JSON retriever:", self.retriever_json_path)
        print("Claims with retrieved evidence:", len(self.retriever_results))

    def retrieve_by_claim_id(self, claim_id: str, top_k: int = 4) -> List[str]:
        if claim_id not in self.retriever_results:
            return []
        return self.retriever_results[claim_id].get("evidences", [])[:top_k]

    def retrieve(self, claim_text: str, top_k: int = 4) -> List[str]:
        # Backup method only. Prefer retrieve_by_claim_id for this assignment.
        return self.text_to_evidences.get(normalise_text(claim_text), [])[:top_k]


train_json_retriever = FixedJSONRetriever(TRAIN_RETRIEVER_JSON_PATH)
dev_json_retriever = FixedJSONRetriever(DEV_RETRIEVER_JSON_PATH)

Loaded JSON retriever: /content/drive/MyDrive/NLP/data/train-retriever-only-k4-bm25200.json
Claims with retrieved evidence: 1228
Loaded JSON retriever: /content/drive/MyDrive/NLP/data/dev-retriever-only-k4-bm25200.json
Claims with retrieved evidence: 154


## 6. Build vocabulary

In [35]:
def build_vocab_with_retrieved_evidence(
    claims_json,
    evidence_corpus,
    retriever,
    retrieval_top_k=4,
    min_freq=2,
    max_vocab_size=50000
):
    counter = Counter()

    for claim_id, instance in claims_json.items():
        # Claim text
        counter.update(simple_tokenise(instance["claim_text"]))

        # Retrieved evidence, not gold evidence
        evidence_ids = retriever.retrieve_by_claim_id(
            claim_id,
            top_k=retrieval_top_k
        )

        for eid in evidence_ids:
            if eid in evidence_corpus:
                counter.update(simple_tokenise(evidence_corpus[eid]))

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1,
        "<CLAIM>": 2,
        "<EVIDENCE>": 3
    }

    for word, freq in counter.most_common(max_vocab_size):
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)

    return vocab

In [36]:
vocab = build_vocab_with_retrieved_evidence(
    claims_json=train_claims,
    evidence_corpus=evidence_corpus,
    retriever=train_json_retriever,
    retrieval_top_k=4,
    min_freq=2,
    max_vocab_size=50000
)

print("Vocab size:", len(vocab))

Vocab size: 6825


## 7. Shared dataset and collate function

In [37]:
def encode_tokens(tokens, vocab, max_len=512):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    return ids[:max_len]


def encode_claim_evidence(claim_text, evidence_text, vocab, max_len=512):
    tokens = (
        ["<CLAIM>"]
        + simple_tokenise(claim_text)
        + ["<EVIDENCE>"]
        + simple_tokenise(evidence_text)
    )
    return encode_tokens(tokens, vocab, max_len=max_len)


class ClaimEvidenceDataset(Dataset):
    def __init__(
        self,
        claims_json,
        evidence_corpus,
        vocab,
        label2id,
        max_len=256,
        max_evidence=4,
        use_gold_evidence=False,
        retriever=None,
        retrieval_top_k=4,
        evidence_mode="retrieved",
        is_test=False
    ):
        self.claims_json = claims_json
        self.items = list(claims_json.items())
        self.claim_ids = [claim_id for claim_id, _ in self.items]
        self.evidence_corpus = evidence_corpus
        self.vocab = vocab
        self.label2id = label2id
        self.max_len = max_len
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.evidence_mode = evidence_mode
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        gold_ids = instance.get("evidences", [])

        retrieved_ids = []
        if self.retriever is not None:
            retrieved_ids = self.retriever.retrieve_by_claim_id(
                claim_id,
                top_k=self.retrieval_top_k
            )

        if self.evidence_mode == "gold":
            evidence_ids = gold_ids

        elif self.evidence_mode == "retrieved":
            evidence_ids = retrieved_ids

        elif self.evidence_mode == "mixed":
            evidence_ids = gold_ids + retrieved_ids

        else:
            raise ValueError(f"Unknown evidence_mode: {self.evidence_mode}")

        # remove duplicated evidence IDs but keep order
        seen = set()
        evidence_ids = [
            eid for eid in evidence_ids
            if not (eid in seen or seen.add(eid))
        ]

        evidence_ids = evidence_ids[:self.max_evidence]

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        input_ids = encode_claim_evidence(
            claim_text=claim_text,
            evidence_text=evidence_text,
            vocab=self.vocab,
            max_len=self.max_len
        )

        item = {
            "claim_id": claim_id,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "evidence_ids": evidence_ids[:self.max_evidence]
        }

        if not self.is_test:
            item["label"] = torch.tensor(LABEL2ID[instance["claim_label"]], dtype=torch.long)

        return item


def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=vocab["<PAD>"])
    attention_mask = (input_ids != vocab["<PAD>"]).long()

    output = {
        "claim_ids": [item["claim_id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": [item["evidence_ids"] for item in batch]
    }

    if "label" in batch[0]:
        output["labels"] = torch.stack([item["label"] for item in batch])

    return output

## 8. Architecture 1: CNN + BiLSTM + attention pooling

In [38]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attention(hidden_states).squeeze(-1)

        if attention_mask is not None:
            scores = scores.masked_fill(~attention_mask.bool(), -1e9)

        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)
        return pooled


class CNNBiLSTMAttentionClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        dropout=0.4,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, cnn_channels, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bilstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_output_dim = lstm_hidden_dim * 2
        self.attention_pooling = AttentionPooling(lstm_output_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        conv_input = embedded.transpose(1, 2)
        conv_outputs = [F.relu(conv(conv_input)) for conv in self.convs]

        min_len = min(x.size(2) for x in conv_outputs)
        conv_outputs = [x[:, :, :min_len] for x in conv_outputs]
        cnn_features = torch.cat(conv_outputs, dim=1).transpose(1, 2)

        if attention_mask is not None:
            attention_mask = attention_mask[:, :min_len]

        lstm_output, _ = self.bilstm(cnn_features)
        pooled = self.attention_pooling(lstm_output, attention_mask)
        return self.classifier(self.dropout(pooled))

## 9. Architecture 2: CNN + BiLSTM + multi-head self-attention

In [39]:
class CNNBiLSTMMultiheadClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, cnn_channels, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bilstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_output_dim = lstm_hidden_dim * 2
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=lstm_output_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(lstm_output_dim)
        self.attention_pooling = AttentionPooling(lstm_output_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        conv_input = embedded.transpose(1, 2)
        conv_outputs = [F.relu(conv(conv_input)) for conv in self.convs]

        min_len = min(x.size(2) for x in conv_outputs)
        conv_outputs = [x[:, :, :min_len] for x in conv_outputs]
        cnn_features = torch.cat(conv_outputs, dim=1).transpose(1, 2)

        if attention_mask is not None:
            attention_mask = attention_mask[:, :min_len]
            key_padding_mask = ~attention_mask.bool()
        else:
            key_padding_mask = None

        lstm_output, _ = self.bilstm(cnn_features)
        attn_output, _ = self.multihead_attn(
            lstm_output,
            lstm_output,
            lstm_output,
            key_padding_mask=key_padding_mask
        )

        attn_output = self.layer_norm(lstm_output + attn_output)
        pooled = self.attention_pooling(attn_output, attention_mask)
        return self.classifier(self.dropout(pooled))

## 10. Architecture 3: CNN + BiGRU + multi-head self-attention

In [40]:
class CNNBiGRUMultiheadClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        gru_hidden_dim=128,
        gru_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, cnn_channels, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bigru = nn.GRU(
            input_size=cnn_output_dim,
            hidden_size=gru_hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0.0
        )

        gru_output_dim = gru_hidden_dim * 2
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=gru_output_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(gru_output_dim)
        self.attention_pooling = AttentionPooling(gru_output_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(gru_output_dim, gru_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        conv_input = embedded.transpose(1, 2)
        conv_outputs = [F.relu(conv(conv_input)) for conv in self.convs]

        min_len = min(x.size(2) for x in conv_outputs)
        conv_outputs = [x[:, :, :min_len] for x in conv_outputs]
        cnn_features = torch.cat(conv_outputs, dim=1).transpose(1, 2)

        if attention_mask is not None:
            attention_mask = attention_mask[:, :min_len]
            key_padding_mask = ~attention_mask.bool()
        else:
            key_padding_mask = None

        gru_output, _ = self.bigru(cnn_features)
        attn_output, _ = self.multihead_attn(
            gru_output,
            gru_output,
            gru_output,
            key_padding_mask=key_padding_mask
        )

        attn_output = self.layer_norm(gru_output + attn_output)
        pooled = self.attention_pooling(attn_output, attention_mask)
        return self.classifier(self.dropout(pooled))

## 11. Shared evaluation, training, and prediction functions

In [41]:
def evaluate_classifier(model, dataloader, device="cpu"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device).long()

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Weighted F1:", round(weighted_f1, 4))
    print("\nClassification report:")
    print(classification_report(
        all_labels,
        all_preds,
        labels=list(range(4)),
        target_names=[ID2LABEL[i] for i in range(4)],
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds, labels=list(range(4))))

    return acc, macro_f1, weighted_f1


def get_class_weights_from_dataset(dataset, num_labels, device):
    labels = []
    for item in dataset:
        labels.append(int(item["label"]))

    labels = np.array(labels)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_labels),
        y=labels
    )
    return torch.tensor(class_weights, dtype=torch.float).to(device)


def train_architecture(
    architecture_name: str,
    model_factory: Callable[[], nn.Module],
    train_claims,
    dev_claims,
    evidence_corpus,
    vocab,
    train_retriever,
    dev_retriever,
    train_with_retrieved_evidence=False,
    dev_with_retrieved_evidence=True,
    use_class_weights=False,
    retrieval_top_k=4,
    epochs=8,
    batch_size=32,
    lr=5e-4,
    max_len=384,
    max_evidence=4,
    device="cpu"
):
    set_seed(42)
    print(f"\n===== Training {architecture_name} =====")

    train_dataset = ClaimEvidenceDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        label2id=LABEL2ID,
        max_len=max_len,
        max_evidence=max_evidence,
        retriever=train_json_retriever,
        retrieval_top_k=retrieval_top_k,
        evidence_mode="mixed",
        is_test=False
    )

    dev_dataset = ClaimEvidenceDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        label2id=LABEL2ID,
        max_len=max_len,
        max_evidence=max_evidence,
        retriever=dev_json_retriever,
        retrieval_top_k=retrieval_top_k,
        evidence_mode="retrieved",
        is_test=False
    )

    generator = torch.Generator()
    generator.manual_seed(42)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        generator=generator
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    model = model_factory().to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    if use_class_weights:
        class_weights = get_class_weights_from_dataset(train_dataset, num_labels=4, device=device)
        print("Class weights:", class_weights)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    best_macro_f1 = -1.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device).long()

            optimiser.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()

            total_loss += loss.item()

        print("Training loss:", round(total_loss / max(len(train_loader), 1), 4))
        dev_acc, dev_macro_f1, dev_weighted_f1 = evaluate_classifier(model, dev_loader, device=device)
        print("Dev evidence mode:", "JSON retrieved" if dev_with_retrieved_evidence else "gold")

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print("New best macro F1:", round(best_macro_f1, 4))

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, best_macro_f1


def predict_claims(
    claims_json,
    evidence_corpus,
    retriever,
    model,
    vocab,
    output_path,
    retrieval_top_k=4,
    max_evidence=4,
    batch_size=32,
    max_len=384,
    is_test=False,
    device="cpu"
):
    dataset = ClaimEvidenceDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        label2id=LABEL2ID,
        max_len=max_len,
        max_evidence=max_evidence,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        evidence_mode="retrieved",
        is_test=is_test
    )

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(batch["claim_ids"], pred_ids, batch["evidence_ids"]):
                predictions[claim_id] = {
                    "claim_text": claims_json[claim_id]["claim_text"],
                    "claim_label": ID2LABEL[pred_id],
                    "evidences": evidence_ids[:max_evidence]
                }

    save_json(predictions, output_path)
    print("Saved predictions to:", output_path)
    return predictions

## retriever coverage check & hit-rate check

In [42]:
def check_retriever_coverage(claims_json, retriever, name, top_k=4):
    total = len(claims_json)
    found = 0
    empty = 0

    for claim_id in claims_json.keys():
        evidence_ids = retriever.retrieve_by_claim_id(claim_id, top_k=top_k)

        if len(evidence_ids) > 0:
            found += 1
        else:
            empty += 1

    print(f"\n{name} retriever coverage")
    print("Total claims:", total)
    print("Claims with retrieved evidence:", found)
    print("Claims with empty evidence:", empty)
    print("Coverage:", round(found / total, 4))

check_retriever_coverage(train_claims, train_json_retriever, "Train", top_k=4)
check_retriever_coverage(dev_claims, dev_json_retriever, "Dev", top_k=4)


Train retriever coverage
Total claims: 1228
Claims with retrieved evidence: 1228
Claims with empty evidence: 0
Coverage: 1.0

Dev retriever coverage
Total claims: 154
Claims with retrieved evidence: 154
Claims with empty evidence: 0
Coverage: 1.0


In [43]:
def retrieval_hit_rate(claims_json, retriever, name, top_k=4):
    total = 0
    hit = 0

    for claim_id, item in claims_json.items():
        gold_evidence = set(item.get("evidences", []))
        retrieved_evidence = set(
            retriever.retrieve_by_claim_id(claim_id, top_k=top_k)
        )

        if len(gold_evidence & retrieved_evidence) > 0:
            hit += 1

        total += 1

    print(f"\n{name} retrieval hit rate")
    print("Hit rate:", round(hit / total, 4))

retrieval_hit_rate(train_claims, train_json_retriever, "Train", top_k=4)
retrieval_hit_rate(dev_claims, dev_json_retriever, "Dev", top_k=4)


Train retrieval hit rate
Hit rate: 0.7223

Dev retrieval hit rate
Hit rate: 0.526


## 12. Train all three architectures with the same JSON retriever

In [44]:
COMMON_TRAIN_ARGS = dict(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    vocab=vocab,
    train_retriever=train_json_retriever,
    dev_retriever=dev_json_retriever,
    train_with_retrieved_evidence=True,
    dev_with_retrieved_evidence=True,
    retrieval_top_k=4,
    epochs=8,
    batch_size=64,
    lr=5e-4,
    max_len=384,
    max_evidence=4,
    device=device
)


def make_cnn_bilstm_attention():
    return CNNBiLSTMAttentionClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        dropout=0.4,
        pad_idx=vocab["<PAD>"]
    )


def make_cnn_bilstm_multihead():
    return CNNBiLSTMMultiheadClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=vocab["<PAD>"]
    )


def make_cnn_bigru_multihead():
    return CNNBiGRUMultiheadClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        gru_hidden_dim=128,
        gru_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=vocab["<PAD>"]
    )


model_1_cnn_bilstm_attention, best_f1_1 = train_architecture(
    architecture_name="Model 1: CNN + BiLSTM + attention pooling",
    model_factory=make_cnn_bilstm_attention,
    use_class_weights=False,
    **COMMON_TRAIN_ARGS
)

model_2_cnn_bilstm_multihead, best_f1_2 = train_architecture(
    architecture_name="Model 2: CNN + BiLSTM + multi-head attention",
    model_factory=make_cnn_bilstm_multihead,
    use_class_weights=False,
    **COMMON_TRAIN_ARGS
)

model_3_cnn_bigru_multihead, best_f1_3 = train_architecture(
    architecture_name="Model 3: CNN + BiGRU + multi-head attention",
    model_factory=make_cnn_bigru_multihead,
    use_class_weights=False,
    **COMMON_TRAIN_ARGS
)

print("\nBest dev macro F1 scores:")
print("Model 1 CNN+BiLSTM+attention:", round(best_f1_1, 4))
print("Model 2 CNN+BiLSTM+multihead:", round(best_f1_2, 4))
print("Model 3 CNN+BiGRU+multihead:", round(best_f1_3, 4))


===== Training Model 1: CNN + BiLSTM + attention pooling =====

Epoch 1/8


  0%|          | 0/20 [00:00<?, ?it/s]

Training loss: 1.3051


  0%|          | 0/3 [00:00<?, ?it/s]

Accuracy: 0.4416
Macro F1: 0.1532
Weighted F1: 0.2705

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.44      1.00      0.61        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.00      0.00      0.00        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.44       154
      macro avg       0.11      0.25      0.15       154
   weighted avg       0.19      0.44      0.27       154

Confusion matrix:
[[68  0  0  0]
 [27  0  0  0]
 [41  0  0  0]
 [18  0  0  0]]
Dev evidence mode: JSON retrieved
New best macro F1: 0.1532

Epoch 2/8


...

Accuracy: 0.3896
Macro F1: 0.2664
Weighted F1: 0.342

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.43      0.38      0.41        68
        REFUTES       0.20      0.11      0.14        27
NOT_ENOUGH_INFO       0.39      0.76      0.52        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.39       154
      macro avg       0.26      0.31      0.27       154
   weighted avg       0.33      0.39      0.34       154

Confusion matrix:
[[26  7 35  0]
 [12  3 12  0]
 [10  0 31  0]
 [12  5  1  0]]
Dev evidence mode: JSON retrieved
New best macro F1: 0.2664

Epoch 8/8


  0%|          | 0/20 [00:00<?, ?it/s]

Training loss: 1.0072


  0%|          | 0/3 [00:00<?, ?it/s]

Accuracy: 0.4156
Macro F1: 0.2573
Weighted F1: 0.3538

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.42      0.59      0.49        68
        REFUTES       0.50      0.04      0.07        27
NOT_ENOUGH_INFO       0.40      0.56      0.47        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.42       154
      macro avg       0.33      0.30      0.26       154
   weighted avg       0.38      0.42      0.35       154

Confusion matrix:
[[40  1 27  0]
 [19  1  7  0]
 [18  0 23  0]
 [18  0  0  0]]
Dev evidence mode: JSON retrieved

Best dev macro F1 scores:
Model 1 CNN+BiLSTM+attention: 0.2554
Model 2 CNN+BiLSTM+multihead: 0.2626
Model 3 CNN+BiGRU+multihead: 0.2664


## 14. Generate dev prediction files for all three models

In [49]:
DEV_PRED_PATH_MODEL_1 = OUTPUT_DIR / "dev_predictions_model1_cnn_bilstm_attention_hybrid_retrieve_train_vocab.json"
DEV_PRED_PATH_MODEL_2 = OUTPUT_DIR / "dev_predictions_model2_cnn_bilstm_multihead_hybrid_retreive_train_vocab.json"
DEV_PRED_PATH_MODEL_3 = OUTPUT_DIR / "dev_predictions_model3_cnn_bigru_multihead_hybrid_retreive_train_vocab.json"

dev_predictions_model_1 = predict_claims(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=dev_json_retriever,
    model=model_1_cnn_bilstm_attention,
    vocab=vocab,
    output_path=DEV_PRED_PATH_MODEL_1,
    retrieval_top_k=4,
    max_evidence=4,
    batch_size=32,
    max_len=384,
    is_test=False,
    device=device
)

dev_predictions_model_2 = predict_claims(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=dev_json_retriever,
    model=model_2_cnn_bilstm_multihead,
    vocab=vocab,
    output_path=DEV_PRED_PATH_MODEL_2,
    retrieval_top_k=4,
    max_evidence=4,
    batch_size=32,
    max_len=384,
    is_test=False,
    device=device
)

dev_predictions_model_3 = predict_claims(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=dev_json_retriever,
    model=model_3_cnn_bigru_multihead,
    vocab=vocab,
    output_path=DEV_PRED_PATH_MODEL_3,
    retrieval_top_k=4,
    max_evidence=4,
    batch_size=32,
    max_len=384,
    is_test=False,
    device=device
)

  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/dev_predictions_model1_cnn_bilstm_attention_hybrid_retrieve_train_vocab.json


  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/dev_predictions_model2_cnn_bilstm_multihead_hybrid_retreive_train_vocab.json


  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/dev_predictions_model3_cnn_bigru_multihead_hybrid_retreive_train_vocab.json


## 15. Evaluate with eval.py

In [50]:
import os
os.chdir("/content/drive/MyDrive/NLP/")

In [51]:
print("single attention:")
!python eval.py --predictions outputs/dev_predictions_model1_cnn_bilstm_attention_hybrid_retrieve_train_vocab.json --groundtruth data/dev-claims.json

print("\nmultihead self-attetion")
!python eval.py --predictions outputs/dev_predictions_model2_cnn_bilstm_multihead_hybrid_retreive_train_vocab.json --groundtruth data/dev-claims.json

print("\nGRU architecture")
!python eval.py --predictions outputs/dev_predictions_model3_cnn_bigru_multihead_hybrid_retreive_train_vocab.json --groundtruth data/dev-claims.json

single attention:
Evidence Retrieval F-score (F)    = 0.20048958977530407
Claim Classification Accuracy (A) = 0.44805194805194803
Harmonic Mean of F and A          = 0.27702081061425643

multihead self-attetion
Evidence Retrieval F-score (F)    = 0.20048958977530407
Claim Classification Accuracy (A) = 0.42207792207792205
Harmonic Mean of F and A          = 0.27184916604053544

GRU architecture
Evidence Retrieval F-score (F)    = 0.20048958977530407
Claim Classification Accuracy (A) = 0.38961038961038963
Harmonic Mean of F and A          = 0.2647443820164186


## 16. Optional test prediction with a test JSON retriever

In [52]:
# Only run this if TEST_RETRIEVER_JSON_PATH exists and has the same format as the dev JSON retriever.
# The dev retriever JSON should not be used for test claims because its claim IDs are dev claim IDs.

if TEST_PATH.exists() and TEST_RETRIEVER_JSON_PATH.exists():
    test_claims = load_json(TEST_PATH)
    test_json_retriever = FixedJSONRetriever(TEST_RETRIEVER_JSON_PATH)

    TEST_PRED_PATH_MODEL_1 = OUTPUT_DIR / "test_predictions_model1_cnn_bilstm_attention_hybrid_retriver_train_vocab.json"
    TEST_PRED_PATH_MODEL_2 = OUTPUT_DIR / "test_predictions_model2_cnn_bilstm_multihead_hybrid_retriver_train_vocab.json"
    TEST_PRED_PATH_MODEL_3 = OUTPUT_DIR / "test_predictions_model3_cnn_bigru_multihead_hybrid_retriever_train_vocab.json"

    test_predictions_model_1 = predict_claims(
        claims_json=test_claims,
        evidence_corpus=evidence_corpus,
        retriever=test_json_retriever,
        model=model_1_cnn_bilstm_attention,
        vocab=vocab,
        output_path=TEST_PRED_PATH_MODEL_1,
        retrieval_top_k=4,
        max_evidence=4,
        batch_size=32,
        max_len=384,
        is_test=True,
        device=device
    )

    test_predictions_model_2 = predict_claims(
        claims_json=test_claims,
        evidence_corpus=evidence_corpus,
        retriever=test_json_retriever,
        model=model_2_cnn_bilstm_multihead,
        vocab=vocab,
        output_path=TEST_PRED_PATH_MODEL_2,
        retrieval_top_k=4,
        max_evidence=4,
        batch_size=32,
        max_len=384,
        is_test=True,
        device=device
    )

    test_predictions_model_3 = predict_claims(
        claims_json=test_claims,
        evidence_corpus=evidence_corpus,
        retriever=test_json_retriever,
        model=model_3_cnn_bigru_multihead,
        vocab=vocab,
        output_path=TEST_PRED_PATH_MODEL_3,
        retrieval_top_k=4,
        max_evidence=4,
        batch_size=32,
        max_len=384,
        is_test=True,
        device=device
    )
else:
    print("No test JSON retriever found. Generate test-retriever-only-k4-bm25200.json before test prediction.")

Loaded JSON retriever: /content/drive/MyDrive/NLP/data/test-retriever-only-k4-bm25200.json
Claims with retrieved evidence: 153


  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/test_predictions_model1_cnn_bilstm_attention_hybrid_retriver_train_vocab.json


  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/test_predictions_model2_cnn_bilstm_multihead_hybrid_retriver_train_vocab.json


  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/drive/MyDrive/NLP/outputs/test_predictions_model3_cnn_bigru_multihead_hybrid_retriever_train_vocab.json
